In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from xgboost import XGBClassifier
import joblib

In [2]:
data=pd.read_csv("kidney3.csv")
data

,Age (yrs),Blood Pressure (mm/Hg),Specific Gravity,Albumin,Sugar,Blood Glucose Random (mgs/dL),Blood Urea (mgs/dL),Serum Creatinine (mgs/dL),Sodium (mEq/L),Potassium (mEq/L),...,Pus Cells: normal,Pus Cell Clumps: present,Bacteria: present,Hypertension: yes,Diabetes Mellitus: yes,Coronary Artery Disease: yes,Appetite: poor,Pedal Edema: yes,Anemia: yes,Chronic Kidney Disease: yes
0,48.0,80.0,1.020,1.0,0.0,121.000000,36.0,1.2,136.0,4.7,...,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0
1,7.0,50.0,1.020,4.0,0.0,90.316581,18.0,0.8,140.0,4.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
2,62.0,80.0,1.010,2.0,3.0,423.000000,53.0,1.8,135.0,4.8,...,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,1.0
3,48.0,70.0,1.005,4.0,0.0,117.000000,56.0,3.8,111.0,2.5,...,0.0,1.0,0.0,1.0,0.0,0.0,1.0,1.0,1.0,1.0
4,51.0,80.0,1.010,2.0,0.0,106.000000,26.0,1.4,140.0,4.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
395,55.0,80.0,1.020,0.0,0.0,140.000000,49.0,0.5,150.0,4.9,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
396,42.0,70.0,1.025,0.0,0.0,75.000000,31.0,1.2,141.0,3.5,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
397,12.0,80.0,1.020,0.0,0.0,100.000000,26.0,0.6,137.0,4.4,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
398,17.0,60.0,1.025,0.0,0.0,114.000000,50.0,1.0,135.0,4.9,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [3]:
x=data.drop("Chronic Kidney Disease: yes",axis=1)
y=data["Chronic Kidney Disease: yes"]


In [4]:
print(y.value_counts())

Chronic Kidney Disease: yes
1.0    250
0.0    150
Name: count, dtype: int64


In [5]:
x_train,x_test,y_train,y_test=train_test_split(x,y,random_state=45,test_size=0.2,stratify=y)


In [6]:
corr = data.corr()

corr["Chronic Kidney Disease: yes"].sort_values(ascending=False)

Chronic Kidney Disease: yes       1.000000
Hypertension: yes                 0.590438
Diabetes Mellitus: yes            0.559060
Albumin                           0.553327
Appetite: poor                    0.393341
Blood Glucose Random (mgs/dL)     0.389736
Blood Urea (mgs/dL)               0.375635
Pedal Edema: yes                  0.375154
Anemia: yes                       0.325396
Serum Creatinine (mgs/dL)         0.297125
Sugar                             0.294555
Blood Pressure (mm/Hg)            0.283378
Pus Cell Clumps: present          0.265313
Coronary Artery Disease: yes      0.236088
Age (yrs)                         0.230780
White Blood Cells (cells/cmm)     0.199693
Bacteria: present                 0.186871
Potassium (mEq/L)                 0.062634
Red Blood Cells: normal          -0.350559
Sodium (mEq/L)                   -0.361208
Pus Cells: normal                -0.387298
Red Blood Cells (millions/cmm)   -0.663130
Specific Gravity                 -0.699821
Packed Cell

In [7]:
xgb = XGBClassifier(
    random_state=45,
    eval_metric='logloss'
)

In [8]:
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [3, 5],
    'learning_rate': [0.01, 0.1],
    'subsample': [0.8, 1.0]
}

In [9]:
grid = GridSearchCV(
    estimator=xgb,
    param_grid=param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=2
)

grid.fit(x_train, y_train)

Fitting 5 folds for each of 16 candidates, totalling 80 fits


,estimator,"XGBClassifier...ree=None, ...)"
,param_grid,"{'learning_rate': [0.01, 0.1], 'max_depth': [3, 5], 'n_estimators': [100, 200], 'subsample': [0.8, 1.0]}"
,scoring,'accuracy'
,n_jobs,-1
,refit,True
,cv,5
,verbose,2
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,objective,'binary:logistic'


In [10]:
print("Best Parameters:")
print(grid.best_params_)

print()

print("Best Cross Validation Accuracy:")
print(grid.best_score_)

Best Parameters:
{'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 100, 'subsample': 0.8}

Best Cross Validation Accuracy:
0.99375


In [11]:
best_model = grid.best_estimator_

y_pred = best_model.predict(x_test)

In [12]:
print("Accuracy:", accuracy_score(y_test, y_pred))

print()

print(confusion_matrix(y_test, y_pred))

print()

print(classification_report(y_test, y_pred))

Accuracy: 1.0

[[30  0]
 [ 0 50]]

              precision    recall  f1-score   support

         0.0       1.00      1.00      1.00        30
         1.0       1.00      1.00      1.00        50

    accuracy                           1.00        80
   macro avg       1.00      1.00      1.00        80
weighted avg       1.00      1.00      1.00        80



In [14]:
# Save the trained model
joblib.dump(best_model, "kidney_model.pkl")

['kidney_model.pkl']